**SALES ANALYSIS ECOMM BRAND**


In [1]:
import pandas as pd
import numpy as np


In [2]:
import os
print(os.getcwd())
print(os.listdir())

C:\Users\Owner\Documents\CUSTOMER BEHAVIOUR PROJECT
['.ipynb_checkpoints', 'sales_marketing_clean.csv', 'Sales_Marketing_Customer_Dataset.csv', 'shopping_behavior_updated (1).csv', 'Untitled.ipynb', 'Untitled1.ipynb']


In [3]:
df = pd.read_csv('Sales_Marketing_Customer_Dataset.csv')

In [4]:
df.head()

,customer_id,gender,age,country,city,signup_date,last_purchase_date,acquisition_channel,device_type,subscription_type,...,support_tickets,refund_requested,delivery_delay_days,payment_method,satisfaction_score,nps_score,marketing_spend_per_user,lifetime_value,last_3_month_purchase_freq,churn
0,10001,Male,52.0,India,Berlin,2022-05-10 00:00:00,2024-12-31 00:00:00,Email,Tablet,Annual,...,0,0,3,UPI,3.0,10,27.56,915.310827,14,0
1,10002,NaN,35.0,Germany,Mumbai,2024-06-16 00:00:00,2024-05-07 00:00:00,Organic,Desktop,Monthly,...,5,0,3,BKash,3.0,7,15.15,2079.960938,11,0
2,10003,Female,27.0,Germany,London,2023-08-23 00:00:00,2024-04-28 00:00:00,Email,Mobile,Annual,...,1,0,2,UPI,5.0,6,13.51,1379.150885,9,0
3,10004,Female,36.0,India,Mumbai,2024-01-28 00:00:00,2023-05-20 00:00:00,Facebook Ads,Tablet,Annual,...,0,0,2,PayPal,4.0,6,25.65,774.652684,7,0
4,10005,Male,29.0,USA,Hamburg,2023-07-21 00:00:00,2024-04-07 00:00:00,Referral,Mobile,Monthly,...,2,1,4,BKash,3.0,1,12.39,87.680409,11,0


In [5]:
df.shape

(15000, 30)

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15000 entries, 0 to 14999
Data columns (total 30 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   customer_id                 15000 non-null  int64  
 1   gender                      14262 non-null  object 
 2   age                         13800 non-null  float64
 3   country                     15000 non-null  object 
 4   city                        15000 non-null  object 
 5   signup_date                 15000 non-null  object 
 6   last_purchase_date          15000 non-null  object 
 7   acquisition_channel         15000 non-null  object 
 8   device_type                 15000 non-null  object 
 9   subscription_type           15000 non-null  object 
 10  is_premium_user             15000 non-null  int64  
 11  total_visits                15000 non-null  int64  
 12  avg_session_time            15000 non-null  float64
 13  pages_per_session           150

In [7]:
# 1. coupon vs discount flag
print(pd.crosstab(df['coupon_code'].isna(), df['discount_used'], margins=True))

# 2. impossible dates
df['signup_date'] = pd.to_datetime(df['signup_date'], errors='coerce')
df['last_purchase_date'] = pd.to_datetime(df['last_purchase_date'], errors='coerce')
bad = df['last_purchase_date'] < df['signup_date']
print(bad.sum(), round(bad.mean()*100, 1))

# 3. LTV below what the customer already spent
print((df['lifetime_value'] < df['total_spent']).sum())

# 4. are the missing total_spent rows different?
print(df.assign(spent_missing=df['total_spent'].isna())
        .groupby('spent_missing')[['churn','is_premium_user','avg_order_value','lifetime_value']].mean())

discount_used     0     1    All
coupon_code                     
False          4537  4330   8867
True           3046  3087   6133
All            7583  7417  15000
3762 25.1
2340
                  churn  is_premium_user  avg_order_value  lifetime_value
spent_missing                                                            
False          0.157204         0.305663        60.086766     1235.889291
True           0.100000         0.288571        60.002712     1233.159419


In [8]:
df_raw = df.copy()

df['gender'] = df['gender'].fillna('Unknown')
df['age_band'] = (pd.cut(df['age'], bins=[0,24,34,44,54,120],
                 labels=['<25','25-34','35-44','45-54','55+'])
                 .astype(object).fillna('Unknown'))

df['spend_recorded'] = df['total_spent'].notna()
df['date_valid'] = df['last_purchase_date'] >= df['signup_date']
df['ltv_not_contradicted'] = df['total_spent'].isna() | (df['lifetime_value'] >= df['total_spent'])
df['coupon_discount_conflict'] = (
    (df['coupon_code'].notna() & (df['discount_used'] == 0)) |
    (df['coupon_code'].isna() & (df['discount_used'] == 1)))
df['signup_month'] = df['signup_date'].dt.to_period('M').astype(str)

df.to_csv('sales_marketing_clean.csv', index=False)

In [9]:
df.head(50)

,customer_id,gender,age,country,city,signup_date,last_purchase_date,acquisition_channel,device_type,subscription_type,...,marketing_spend_per_user,lifetime_value,last_3_month_purchase_freq,churn,age_band,spend_recorded,date_valid,ltv_not_contradicted,coupon_discount_conflict,signup_month
0,10001,Male,52.0,India,Berlin,2022-05-10,2024-12-31,Email,Tablet,Annual,...,27.56,915.310827,14,0,45-54,True,True,True,True,2022-05
1,10002,Unknown,35.0,Germany,Mumbai,2024-06-16,2024-05-07,Organic,Desktop,Monthly,...,15.15,2079.960938,11,0,35-44,True,False,True,False,2024-06
2,10003,Female,27.0,Germany,London,2023-08-23,2024-04-28,Email,Mobile,Annual,...,13.51,1379.150885,9,0,25-34,True,True,True,False,2023-08
3,10004,Female,36.0,India,Mumbai,2024-01-28,2023-05-20,Facebook Ads,Tablet,Annual,...,25.65,774.652684,7,0,35-44,True,False,True,False,2024-01
4,10005,Male,29.0,USA,Hamburg,2023-07-21,2024-04-07,Referral,Mobile,Monthly,...,12.39,87.680409,11,0,25-34,True,True,False,True,2023-07
5,10006,Male,29.0,Germany,Mumbai,2024-05-08,2023-01-20,Google Ads,Mobile,Monthly,...,16.30,467.452413,14,0,25-34,True,False,True,True,2024-05
6,10007,Male,39.0,UK,New York,2022-01-04,2023-12-05,Email,Mobile,Monthly,...,29.76,1697.386983,1,0,35-44,True,True,True,False,2022-01
7,10008,Female,NaN,Germany,Berlin,2023-05-11,2024-02-14,Facebook Ads,Desktop,Annual,...,18.86,1337.967629,6,0,Unknown,True,True,True,False,2023-05
8,10009,Female,21.0,USA,London,2023-01-14,2024-08-19,Organic,Desktop,Monthly,...,27.14,648.012150,2,1,<25,True,True,True,True,2023-01
9,10010,Female,35.0,Germany,Hamburg,2022-12-22,2024-09-04,Email,Tablet,Monthly,...,5.22,1360.483364,9,1,35-44,True,True,True,True,2022-12


In [10]:
print(df.groupby('acquisition_channel')[['churn','nps_score','marketing_spend_per_user','total_spent']].mean())

                        churn  nps_score  marketing_spend_per_user  \
acquisition_channel                                                  
Email                0.149676   4.987726                 17.590102   
Facebook Ads         0.148148   4.890873                 17.689795   
Google Ads           0.151625   4.926813                 17.552983   
Organic              0.158756   5.019313                 17.238962   
Referral             0.157769   5.020401                 17.814991   

                     total_spent  
acquisition_channel               
Email                 524.940766  
Facebook Ads          524.030649  
Google Ads            528.704138  
Organic               530.680229  
Referral              512.999555  


In [11]:
num = df.select_dtypes('number').corr()['churn'].drop('churn').sort_values()
print(num.round(3))

satisfaction_score           -0.304
total_spent                  -0.163
email_open_rate              -0.008
marketing_spend_per_user     -0.007
refund_requested             -0.007
is_premium_user              -0.005
email_click_rate             -0.002
avg_order_value              -0.001
discount_used                 0.000
last_3_month_purchase_freq    0.002
delivery_delay_days           0.002
age                           0.005
nps_score                     0.006
pages_per_session             0.006
lifetime_value                0.006
avg_session_time              0.012
customer_id                   0.012
total_visits                  0.013
support_tickets               0.128
Name: churn, dtype: float64


In [12]:
# churn by satisfaction score
print(df.groupby('satisfaction_score')['churn'].agg(['mean','count']).round(3))

# churn by support tickets (cap at 5+)
print(df.groupby(df['support_tickets'].clip(upper=5))['churn'].agg(['mean','count']).round(3))

# churn by spend band
print(df.groupby(pd.qcut(df['total_spent'], 4, duplicates='drop'), observed=True)['churn'].agg(['mean','count']).round(3))

# does NPS relate to satisfaction?
print(df[['nps_score','satisfaction_score']].corr().round(3))

                     mean  count
satisfaction_score              
1.0                 0.487    753
2.0                 0.502   1455
3.0                 0.097   3739
4.0                 0.089   5323
5.0                 0.097   3028
                  mean  count
support_tickets              
0                0.127   2071
1                0.142   4047
2                0.132   4069
3                0.132   2653
4                0.121   1351
5                0.504    809
                       mean  count
total_spent                       
(0.266, 300.432]      0.342   3488
(300.432, 498.844]    0.104   3487
(498.844, 702.397]    0.089   3487
(702.397, 15910.432]  0.094   3488
                    nps_score  satisfaction_score
nps_score               1.000              -0.009
satisfaction_score     -0.009               1.000


In [13]:
df['gender'] = df['gender'].fillna('Unknown')
df['spend_recorded'] = df['total_spent'].notna()
df['date_valid'] = df['last_purchase_date'] >= df['signup_date']
df['signup_month'] = df['signup_date'].dt.to_period('M').astype(str)
df.to_csv('sales_marketing_clean.csv', index=False)
print("Saved!", df.shape)

Saved! (15000, 36)


In [14]:
spent = df.loc[df['spend_recorded'], 'total_spent']

print("Customers:", len(df))
print("Total revenue:", round(spent.sum(), 2))
print("ARPU:", round(spent.mean(), 2))
print("AOV:", round(df['avg_order_value'].mean(), 2))
print("CAC:", round(df['marketing_spend_per_user'].mean(), 2))
print("LTV (as provided):", round(df['lifetime_value'].mean(), 2))
print("Churn rate:", round(df['churn'].mean() * 100, 1), "%")
print("Retention rate:", round((1 - df['churn'].mean()) * 100, 1), "%")
print("Premium conversion:", round(df['is_premium_user'].mean() * 100, 1), "%")

promoters = (df['nps_score'] >= 9).mean()
detractors = (df['nps_score'] <= 6).mean()
print("NPS:", round((promoters - detractors) * 100, 1))

Customers: 15000
Total revenue: 7314785.69
ARPU: 524.36
AOV: 60.08
CAC: 17.58
LTV (as provided): 1235.7
Churn rate: 15.3 %
Retention rate: 84.7 %
Premium conversion: 30.4 %
NPS: -46.2


In [15]:
print(df['age'].describe())

count    13800.000000
mean        35.203913
std         10.334384
min         -4.000000
25%         28.000000
50%         35.000000
75%         42.000000
max         95.000000
Name: age, dtype: float64


In [16]:
bins = [0, 24, 34, 54, 120]
labels = ['Youth (under 25)', 'Young Adults (25-34)', 'Adults (35-54)', 'Senior Adults (55+)']

df['age_band'] = (pd.cut(df['age'], bins=bins, labels=labels)
                    .astype(object).fillna('Unknown'))

order = {'Youth (under 25)': 1, 'Young Adults (25-34)': 2,
         'Adults (35-54)': 3, 'Senior Adults (55+)': 4, 'Unknown': 5}
df['age_band_order'] = df['age_band'].map(order)

print(df['age_band'].value_counts())
df.to_csv('sales_marketing_clean.csv', index=False)

age_band
Adults (35-54)          6851
Young Adults (25-34)    4586
Youth (under 25)        1980
Unknown                 1204
Senior Adults (55+)      379
Name: count, dtype: int64


In [17]:
df.groupby('age_band')['total_spent'].median()

age_band
Adults (35-54)          494.852556
Senior Adults (55+)     539.848238
Unknown                 476.385182
Young Adults (25-34)    506.649220
Youth (under 25)        499.529315
Name: total_spent, dtype: float64

In [18]:
s = df[df['spend_recorded']]
print(s.groupby('age_band')['total_spent'].agg(['count','mean','median','max']).round(1))

cap = s['total_spent'].quantile(0.99)
print(s.assign(capped=s['total_spent'].clip(upper=cap)).groupby('age_band')['capped'].mean().round(1))

                      count    mean  median      max
age_band                                            
Adults (35-54)         6350   507.0   494.9   1404.8
Senior Adults (55+)     355  1079.0   539.8  15910.4
Unknown                1116   495.3   476.4   1539.3
Young Adults (25-34)   4276   515.5   506.6   1475.0
Youth (under 25)       1853   515.5   499.5   1489.3
age_band
Adults (35-54)          506.5
Senior Adults (55+)     569.1
Unknown                 494.7
Young Adults (25-34)    514.6
Youth (under 25)        514.6
Name: capped, dtype: float64


In [19]:
print(df['nps_score'].value_counts().sort_index())
print(df['satisfaction_score'].mean().round(2))

nps_score
0     1370
1     1346
2     1417
3     1402
4     1388
5     1303
6     1386
7     1350
8     1356
9     1357
10    1325
Name: count, dtype: int64
3.59
